# 02 — Logistic regression baseline

This project uses synthetic data inspired by common online gaming analytics use cases. It does not contain confidential or proprietary Junglee Games data.

A linear model is the baseline a stakeholder can audit. It is trained on the April snapshot and scored on June, with the decision threshold chosen on May. Preprocessing is median imputation, standardization, and one-hot encoding, fit inside the pipeline so the holdout does not set the medians.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Project root:", ROOT)


Project root: /workspace/player-churn-prediction


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from src.preprocessing import make_one_hot_preprocessor, model_feature_frame
from src.evaluation import classification_metrics, tune_threshold

frame = pd.read_csv(ROOT / "data/processed/churn_dataset.csv")
splits = {}
for name in ("train", "valid", "test"):
    part = frame.loc[frame["dataset_split"] == name].reset_index(drop=True)
    splits[name] = (model_feature_frame(part), part["churned"].to_numpy().astype(int))

pipe = Pipeline([
    ("pre", make_one_hot_preprocessor(scale_numeric=True)),
    ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=500, solver="lbfgs")),
])
pipe.fit(splits["train"][0], splits["train"][1])
p_valid = pipe.predict_proba(splits["valid"][0])[:, 1]
p_test = pipe.predict_proba(splits["test"][0])[:, 1]
chosen = tune_threshold(splits["valid"][1], p_valid, beta=1.5)
metrics = classification_metrics(splits["test"][1], p_test, chosen["threshold"])
at_half = classification_metrics(splits["test"][1], p_test, 0.5)
display(pd.DataFrame([
    {"operating_point": "May-tuned F1.5", **{k: metrics[k] for k in ("threshold","accuracy","precision","recall","f1","roc_auc","pr_auc")}},
    {"operating_point": "0.50", **{k: at_half[k] for k in ("threshold","accuracy","precision","recall","f1","roc_auc","pr_auc")}},
]))
base = float(splits["test"][1].mean())
display(Markdown(
    f"June base rate is **{base:.1%}**. Accuracy at the tuned threshold is **{metrics['accuracy']:.1%}**. "
    f"A classifier that calls every player retained scores about **{1-base:.1%}** accuracy and **zero** recall. "
    f"This baseline's recall is **{metrics['recall']:.1%}** and PR-AUC is **{metrics['pr_auc']:.3f}** "
    f"against a base-rate PR-AUC of {base:.3f}. "
    "Recall matters because a missed churner is a player the desk never called. "
    "PR-AUC matters because it judges the ranking across thresholds when the positive class is the minority. "
    "Accuracy mostly restates the base rate."
))
cm = metrics["confusion_matrix"]
display(Markdown(
    f"Confusion matrix at the tuned threshold — true negatives {cm['tn']:,}, "
    f"false positives {cm['fp']:,} (contacted a player who stayed), "
    f"false negatives {cm['fn']:,} (missed a churner), true positives {cm['tp']:,}."
))


,operating_point,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,May-tuned F1.5,0.6400,0.6422,0.3451,0.7276,0.4681,0.7158,0.3674
1,0.50,0.5000,0.5817,0.3149,0.7938,0.4510,0.7158,0.3674


June base rate is **21.6%**. Accuracy at the tuned threshold is **64.2%**. A classifier that calls every player retained scores about **78.4%** accuracy and **zero** recall. This baseline's recall is **72.8%** and PR-AUC is **0.367** against a base-rate PR-AUC of 0.216. Recall matters because a missed churner is a player the desk never called. PR-AUC matters because it judges the ranking across thresholds when the positive class is the minority. Accuracy mostly restates the base rate.

Confusion matrix at the tuned threshold — true negatives 34,498, false positives 21,267 (contacted a player who stayed), false negatives 4,194 (missed a churner), true positives 11,205.